In [ ]:
import torch
from rdkit import Chem, RDLogger
from src.mlconfgen import MLConformerGenerator, evaluate_samples
from src.mlconfgen.utils import standardize_mol
from pharmacophore import color_tanimoto

RDLogger.DisableLog('rdApp.*')

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps:0")
else:
    device = torch.device("cpu")

print(f"Intitialising model on {device}")

generator = MLConformerGenerator(
                                 edm_weights="./edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="./adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=20,
                                )

ref_mol = Chem.MolFromMolFile('./assets/demo_files/yibfeu.mol')

def score(mol):
    try:
        mol = standardize_mol(mol, optimize_geometry=True)
        ref_mb, scores = evaluate_samples(ref_mol, [mol])
        aligned_ref = Chem.MolFromMolBlock(ref_mb)
        aligned_cand = Chem.MolFromMolBlock(scores[0]['mol_block'])

        color_sim = color_tanimoto(aligned_ref, aligned_cand)
        return color_sim
    except:
        return 0
    

generator.fine_tune(
                  score_function=score,  # This should output normalised score from (0, 1)
                  reference_conformer=ref_mol,
                  variance= 1,
                  # RL Fine-tune params
                  n_epochs=100,
                  train_batch_size=4,
                  learning_rate= 8e-5,
                  sigma=60.0,
                  temperature=1.5,
                  n_samples_per_mol=8,
                  reward_clip=(-1.0, 1.0),
                  eval_every=2,
                  save_dir="./rl_checkpoints_validity",
    
)



[18:10:03] Initializing Normalizer


Intitialising model on mps:0


2026-04-02 18:10:22 | INFO | [Epoch 0001/0100] loss=1202.1676 mean_score=-0.1986 valid_rate=0.6875 agent_ll=-2.3031 prior_ll=-2.3031
2026-04-02 18:10:34 | INFO | [Epoch 0002/0100] loss=2723.1321 mean_score=-0.7128 valid_rate=0.2500 agent_ll=-3.0236 prior_ll=-3.0208
2026-04-02 18:12:33 | INFO | Evaluation:
                             agent_score_mean=0.1049  baseline_score_mean=0.1000
                             eval_agent_valid_rate=0.6406  eval_baseline_valid_rate=0.5469
                             score_improv=0.0049  valid_rate_improv=0.0938 
2026-04-02 18:12:34 | INFO | Saved new best checkpoint
2026-04-02 18:12:47 | INFO | [Epoch 0003/0100] loss=1942.1240 mean_score=-0.4922 valid_rate=0.4688 agent_ll=-2.7796 prior_ll=-3.5225
2026-04-02 18:13:03 | INFO | [Epoch 0004/0100] loss=1532.3706 mean_score=-0.3247 valid_rate=0.5938 agent_ll=-3.5045 prior_ll=-3.7017
2026-04-02 18:14:58 | INFO | Evaluation:
                             agent_score_mean=0.0776  baseline_score_mean=0.0744
  

In [ ]:
import time
import torch

from rdkit import Chem, RDLogger
from rdkit.Chem import Draw

from mlconfgen import MLConformerGenerator, evaluate_samples



generator = MLConformerGenerator(
                                 edm_weights="./edm_moi_chembl_15_39.pt",
                                 adj_mat_seer_weights="./adj_mat_seer_chembl_15_39.pt",
                                 device=device,
                                 diffusion_steps=20,
                                )

# generator.adj_mat_seer.resize.load_state_dict(torch.load("./rl_checkpoints_long_run_2/best_agent_resize.pt"))

# Load a Reference conformer
ref_mol = Chem.MolFromMolFile('./assets/demo_files/yibfeu.mol')


# Generate Samples
print("Generation started...")
start = time.time()

# Resampling significantly increases generation quality, while sacrificing speed
N_SAMPLES = 20
samples = generator.generate_conformers(
                                        reference_conformer=ref_mol,
                                        n_samples=N_SAMPLES,
                                        variance=1,
                                        resample_steps=0,
                                        )

print(f"Generation complete in {round(time.time() - start, 2)}")

# Characterise samples   
_, std_samples = evaluate_samples(ref_mol, samples)

# Display results
mols = []
legends = []
average_shape_similarity = 0
for sample in std_samples:
    mol = Chem.MolFromMolBlock(sample['mol_block'])
    mol = Chem.MolFromSmiles(Chem.MolToSmiles(mol))
    mol.SetProp("Shape_Tanimoto", str(sample['shape_tanimoto']))
    mols.append(mol)
    legends.append(f"Shape Similarity - {round(sample['shape_tanimoto'], 2)}")
    average_shape_similarity += round(sample['shape_tanimoto'], 2)

average_shape_similarity = average_shape_similarity / len(std_samples)
print(f"AVERAGE SHAPE SIMILARITY - {average_shape_similarity}")
print(f"VALID SAMPLES - {round(len(std_samples) / N_SAMPLES, 2) * 100} %")
    
Draw.MolsToGridImage(mols, legends=legends)